# Qwen-Image-2.1-Turbo 生图（Colab 免费 T4）

文生图 + 图生图（改图、最多 10 张参考图），生成后自动传到小橘云盘 `/生图/`。

**用之前：**
1. 菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**。
2. 左边钥匙图标「Secrets」里加 `XIAOJU_TOKEN`（小橘云盘有写权限的 xa_ 令牌），并打开「笔记本访问权限」。不想上传就不用加。
3. 从上往下依次运行。第一次要下载约 23GB 模型，几分钟。

**配置：**
- 去噪模型：`unsloth/Qwen-Image-2.1-Turbo-GGUF` 的 Q4_K_M（4.2GB，8 步出图）
- 文本编码器 Qwen3-VL-8B：官方权重，加载时压成 4-bit（约 5.5GB 显存）

**注意：**
- 许可证是 Qwen Research License：只能研究和非商业用途，商用要另找阿里授权。
- 官方开源权重本身没有内容过滤器，所以不需要「无审查版」。但涉及未成年人的性内容、未经本人同意的真人色情 / 换脸，在几乎所有地方都违法，别生成。
- 免费版 Colab 不要挂 ComfyUI / WebUI 网页界面或隧道，会被警告断开；这个 notebook 只用代码生图。

In [ ]:
# 1. 安装依赖（约 2 分钟）。Qwen-Image-2.1 需要 diffusers 主分支和 transformers>=5.17
!pip -q install git+https://github.com/huggingface/diffusers.git
!pip -q install "transformers>=5.17.0" accelerate gguf bitsandbytes pillow requests
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 配置
REPO = "Qwen/Qwen-Image-2.1-Turbo"          # 文本编码器、VAE、调度器从这里拿
GGUF_REPO = "unsloth/Qwen-Image-2.1-Turbo-GGUF"
QUANT = "Q4_K_M"   # 显存不够换 Q3_K_XL；想要更接近原版画质换 Q5_K_M / Q6_K（显存更紧）
RES = 1024         # 输出边长基准。T4 上 1024 比较稳；1536 以上容易爆显存
UPLOAD_DIR = "/生图"   # 小橘云盘目标文件夹；设成 None 就不上传

In [ ]:
# 3. 加载模型（第一次要下载约 23GB）
import os, gc, torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from huggingface_hub import hf_hub_download
from diffusers import QwenImage21Pipeline, QwenImage21Transformer2DModel, GGUFQuantizationConfig
from transformers import Qwen3VLForConditionalGeneration, BitsAndBytesConfig

DTYPE = torch.bfloat16

gguf_path = hf_hub_download(GGUF_REPO, f"qwen-image-2.1-turbo-{QUANT}.gguf")
transformer = QwenImage21Transformer2DModel.from_single_file(
    gguf_path,
    quantization_config=GGUFQuantizationConfig(compute_dtype=DTYPE),
    config=REPO, subfolder="transformer",
    dtype=DTYPE,
)

text_encoder = Qwen3VLForConditionalGeneration.from_pretrained(
    REPO, subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        llm_int8_skip_modules=["visual", "lm_head"],  # 看图的部分不压，改图更准
    ),
    dtype=DTYPE, device_map={"": 0},
)

pipe = QwenImage21Pipeline.from_pretrained(
    REPO, transformer=transformer, text_encoder=text_encoder, dtype=DTYPE,
)
pipe.to("cuda")
try:
    pipe.vae.enable_tiling()   # 解码分块，省显存
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
print(f"加载完成，显存已用 {torch.cuda.memory_allocated() / 2**30:.1f} GB")

In [ ]:
# 4. 生图 / 上传函数
import io, time, urllib.parse, requests
from PIL import Image
from IPython.display import display

# 官方推荐比例（按 2048 基准），会按 RES 等比缩小
RATIOS = {"1:1": (2048, 2048), "4:3": (2400, 1792), "3:4": (1792, 2400), "3:2": (2528, 1696),
          "2:3": (1696, 2528), "16:9": (2752, 1536), "9:16": (1536, 2752)}

try:
    from google.colab import userdata
    XIAOJU_TOKEN = userdata.get("XIAOJU_TOKEN")
except Exception:
    XIAOJU_TOKEN = None
XIAOJU_API = "https://langhua1998-xiaoju-cloud.hf.space/api/agent/v1"
os.makedirs("/content/outputs", exist_ok=True)

def upload(local_path):
    if not (UPLOAD_DIR and XIAOJU_TOKEN):
        print("（没配 XIAOJU_TOKEN 或 UPLOAD_DIR，不上传）"); return
    dest = f"{UPLOAD_DIR}/{os.path.basename(local_path)}"
    with open(local_path, "rb") as f:
        r = requests.put(f"{XIAOJU_API}/files", params={"path": dest}, data=f.read(), timeout=300,
                         headers={"Authorization": f"Bearer {XIAOJU_TOKEN}",
                                  "Content-Type": "application/octet-stream"})
    print(f"已上传 小橘云盘 {dest}" if r.ok else f"上传失败 {r.status_code}: {r.text[:200]}")

def gen(prompt, images=None, ratio=None, seed=None, negative_prompt=None, cfg=1.0, name="img"):
    """images=None 文生图；传 PIL 图片或图片列表（最多 10 张）就是图生图 / 改图。
    ratio 不填：文生图默认 1:1，图生图跟随最后一张参考图的比例。"""
    seed = int(time.time()) % 2**31 if seed is None else seed
    kw = {}
    if ratio or images is None:
        w, h = RATIOS[ratio or "1:1"]
        kw["width"], kw["height"] = w * RES // 2048, h * RES // 2048
    if images is not None:
        images = images if isinstance(images, list) else [images]
        # T4 显存小：参考图先缩到 RES 以内
        images = [im.copy() for im in images]
        for im in images:
            im.thumbnail((RES, RES))
        kw["image"] = images
    if negative_prompt:
        kw["negative_prompt"], kw["true_cfg_scale"] = negative_prompt, cfg
    t = time.time()
    out = pipe(prompt=prompt, output_resolution=RES, use_kv_cache=True,
               generator=torch.Generator("cpu").manual_seed(seed), **kw).images[0]
    path = f"/content/outputs/{name}_{time.strftime('%Y%m%d_%H%M%S')}_{seed}.png"
    out.save(path)
    print(f"{out.size[0]}x{out.size[1]}  seed={seed}  {time.time() - t:.0f} 秒  → {path}")
    display(out)
    upload(path)
    torch.cuda.empty_cache()
    return out

## 文生图
`ratio` 可选 1:1 / 4:3 / 3:4 / 3:2 / 2:3 / 16:9 / 9:16。中文、英文提示词都行，写得越具体越好。想复现同一张图就固定 `seed`。

In [ ]:
img = gen("一只橘猫趴在窗台上晒太阳，水彩画风格，暖色调，窗外是樱花", ratio="3:4")

## 图生图 / 改图
运行后会弹出上传按钮，可以一次选多张（最多 10 张，T4 上建议 1–2 张）。
提示词写要怎么改，比如「把背景换成海边」「衣服改成红色」「把图1的人放进图2的场景里」。

In [ ]:
from google.colab import files
uploaded = files.upload()
refs = [Image.open(io.BytesIO(b)).convert("RGBA") for b in uploaded.values()]
print(f"收到 {len(refs)} 张参考图")

In [ ]:
img = gen("把背景换成黄昏的海边，保持人物不变", images=refs, name="edit")

## 透明背景
官方推荐的写法：提示词开头写 `This is an RGBA image with transparency.`，结尾写 `The image has alpha channel and the background is transparent.`，中间写要画什么。保存的 PNG 带透明通道，适合做贴纸、图标。

In [ ]:
img = gen("This is an RGBA image with transparency. 一个可爱的橘子卡通贴纸，粗描边。The image has alpha channel and the background is transparent.", name="sticker")

## 常见问题
- **CUDA out of memory**：把 `RES` 改成 896 或 768，或者 `QUANT` 换成 `Q3_K_XL`，然后「代码执行程序 → 重启」，从第 2 格重新跑。
- **一直分不到 GPU**：免费额度用完了，过几小时再试，或者升级 Colab Pro。
- **断开后**：环境清空，要从第 1 格重新跑；模型会重新下载。
- 生成的图都在 `/content/outputs/`，左边「文件」里可以下载；配了令牌的话云盘 `/生图/` 里也有。